# Chemical Structures, Biological Targets, and Activity Measurements for INCITES Compounds Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, inspecting, and exploring a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.nnag-dag9/fair2.json`

This dataset describes an integrated chemical and bioactivity resource for INCITES compounds, including metadata about structures, categories, targets, activity measurements, descriptors, and analysis outputs.

> Note: In this particular Croissant document, the top-level metadata indicates that `recordSet` is empty. This notebook therefore demonstrates both a standard `mlcroissant` workflow for checking available record sets by `@id` and a metadata-first exploration workflow when no record sets are exposed in the schema.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn requests

## 1. Data Loading
Load the dataset metadata from the Croissant schema and inspect the main descriptive fields.

A key rule in Croissant-based exploration is to reference entities by their `@id` values whenever possible. In this notebook, we will use variables to store important paths and identifiers.

In [ ]:
import json
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import mlcroissant as mlc

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.nnag-dag9/fair2.json"

# Load the dataset through mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a concise dataset summary without subscripting the metadata object directly
print("Dataset loaded successfully")
print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

In [ ]:
# Also fetch the raw JSON-LD so we can inspect @id-based structures directly.
response = requests.get(croissant_url, timeout=60)
response.raise_for_status()
raw_croissant = response.json()

print("Top-level keys in the Croissant JSON-LD:")
print(list(raw_croissant.keys()))

dataset_id = raw_croissant.get("@id")
dataset_type = raw_croissant.get("@type")
record_sets = raw_croissant.get("recordSet", [])
distributions = raw_croissant.get("distribution", [])
visualizations = raw_croissant.get("visualization", [])

print("\nTop-level dataset @id:")
print(dataset_id)
print("\nTop-level dataset @type:")
print(dataset_type)
print("\nNumber of record sets declared:", len(record_sets))
print("Number of distributions declared:", len(distributions))
print("Number of visualizations declared:", len(visualizations))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The normal `mlcroissant` workflow is to inspect `recordSet` entities and then load records from a specific record set using its `@id`. Because this schema currently exposes no record sets, we will:

1. Confirm whether any record sets are present.
2. List other available entities by `@id`, such as distributions.
3. Prepare a fallback metadata exploration workflow.

This keeps the notebook faithful to the Croissant structure while remaining robust.

In [ ]:
# Inspect record sets directly from the raw JSON-LD using their @id fields
if record_sets:
    print("Available record sets:")
    for record_set in record_sets:
        record_set_id = record_set.get("@id", "<missing @id>")
        record_set_name = record_set.get("name", "<missing name>")
        print(f"- {record_set_id} | {record_set_name}")
else:
    print("No record sets are declared in this Croissant schema.")
    print("As a result, dataset.records(record_set=<@id>) cannot be demonstrated on a real record set for this dataset.")

print("\nAvailable distribution @id values:")
for dist in distributions:
    print("-", dist.get("@id", "<missing @id>"))

In [ ]:
# Standard mlcroissant pattern for iterating through records from a record set.
# This is only executed if a record set @id exists.

if record_sets:
    example_record_set_id = record_sets[0].get("@id")
    print(f"Previewing records from record set: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("Skipping dataset.records(...) preview because no recordSet @id is available in the schema.")

### Metadata inventory by `@id`
Even when no record sets are available, the Croissant JSON-LD still contains many entities that can be explored by identifier. Below, we summarize key `@id`-based references from the top-level metadata.

In [ ]:
# Collect top-level @id references into convenient tables

author_refs = raw_croissant.get("author", [])
citation_refs = raw_croissant.get("citation", [])

authors_df = pd.DataFrame([
    {"entity_type": "author", "@id": item.get("@id")}
    for item in author_refs
])

citations_df = pd.DataFrame([
    {"entity_type": "citation", "@id": item.get("@id")}
    for item in citation_refs
])

distributions_df = pd.DataFrame([
    {"entity_type": "distribution", "@id": item.get("@id")}
    for item in distributions
])

print("Authors:")
display(authors_df)

print("Citations (first 10 shown):")
display(citations_df.head(10))

print("Distributions:")
display(distributions_df)

## 3. Data Extraction
Load data from available Croissant entities into pandas DataFrames for analysis.

Since there are no `recordSet` entities in this schema, we will extract structured top-level metadata into analysis-ready DataFrames. This is a practical adaptation of the usual record-set loading pattern.

In [ ]:
# Extract data into DataFrames using @id-aware metadata structures

dataframes = {}

# Store top-level repeated metadata sections as DataFrames
dataframes["author_refs"] = authors_df.copy()
dataframes["citation_refs"] = citations_df.copy()
dataframes["distribution_refs"] = distributions_df.copy()

# Convert list-valued descriptive metadata into tabular form
dataframes["keywords"] = pd.DataFrame({"keyword": raw_croissant.get("keywords", [])})
dataframes["data_biases"] = pd.DataFrame({"data_bias": raw_croissant.get("dataBiases", [])})
dataframes["data_limitations"] = pd.DataFrame({"data_limitation": raw_croissant.get("dataLimitations", [])})
dataframes["data_use_cases"] = pd.DataFrame({"data_use_case": raw_croissant.get("dataUseCases", [])})
dataframes["data_collection_type"] = pd.DataFrame({"data_collection_type": raw_croissant.get("dataCollectionType", [])})
dataframes["data_preprocessing_protocol"] = pd.DataFrame({"preprocessing_step": raw_croissant.get("dataPreprocessingProtocol", [])})
dataframes["data_release_maintenance_plan"] = pd.DataFrame({"maintenance_item": raw_croissant.get("dataReleaseMaintenancePlan", [])})

# Build a compact dataset profile DataFrame
dataset_profile = {
    "dataset_@id": raw_croissant.get("@id"),
    "name": raw_croissant.get("name"),
    "identifier": raw_croissant.get("identifier"),
    "version": raw_croissant.get("version"),
    "license": raw_croissant.get("license"),
    "datePublished": raw_croissant.get("datePublished"),
    "num_authors": len(raw_croissant.get("author", [])),
    "num_citations": len(raw_croissant.get("citation", [])),
    "num_distributions": len(raw_croissant.get("distribution", [])),
    "num_record_sets": len(raw_croissant.get("recordSet", [])),
    "num_keywords": len(raw_croissant.get("keywords", [])),
    "num_data_biases": len(raw_croissant.get("dataBiases", [])),
    "num_data_limitations": len(raw_croissant.get("dataLimitations", [])),
    "num_data_use_cases": len(raw_croissant.get("dataUseCases", []))
}
dataframes["dataset_profile"] = pd.DataFrame([dataset_profile])

print("Available extracted tables:")
print(list(dataframes.keys()))

print("\nDataset profile columns:")
print(dataframes["dataset_profile"].columns.tolist())

display(dataframes["dataset_profile"])
display(dataframes["keywords"].head())

### Optional standard record-set extraction pattern
The next code block mirrors the standard template and uses `@id` values for record sets. It will only run if record sets are present.

In [ ]:
# Standard record-set extraction template using record set @id values
# This dataset currently has no record sets, but the pattern is included for completeness.

record_set_ids = [rs.get("@id") for rs in record_sets if rs.get("@id")]
recordset_dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    recordset_dataframes[record_set_id] = pd.DataFrame(records)

if recordset_dataframes:
    first_record_set_id = next(iter(recordset_dataframes))
    print(recordset_dataframes[first_record_set_id].columns.tolist())
    display(recordset_dataframes[first_record_set_id].head())
else:
    print("No record-set DataFrames were created because the schema exposes no recordSet @id values.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the extracted metadata tables.

Because this Croissant schema does not provide row-level record sets, our EDA focuses on structured metadata analysis. We will:

- summarize repeated metadata sections,
- filter items based on simple criteria,
- normalize a numeric summary field,
- group data by derived categories.

In [ ]:
# Build a summary table with numeric counts that we can analyze
summary_counts = pd.DataFrame([
    {"section": "authors", "count": len(raw_croissant.get("author", []))},
    {"section": "citations", "count": len(raw_croissant.get("citation", []))},
    {"section": "distributions", "count": len(raw_croissant.get("distribution", []))},
    {"section": "record_sets", "count": len(raw_croissant.get("recordSet", []))},
    {"section": "keywords", "count": len(raw_croissant.get("keywords", []))},
    {"section": "data_biases", "count": len(raw_croissant.get("dataBiases", []))},
    {"section": "data_limitations", "count": len(raw_croissant.get("dataLimitations", []))},
    {"section": "data_use_cases", "count": len(raw_croissant.get("dataUseCases", []))},
    {"section": "data_collection_type", "count": len(raw_croissant.get("dataCollectionType", []))},
    {"section": "preprocessing_steps", "count": len(raw_croissant.get("dataPreprocessingProtocol", []))},
    {"section": "maintenance_items", "count": len(raw_croissant.get("dataReleaseMaintenancePlan", []))}
])

dataframes["summary_counts"] = summary_counts
display(summary_counts)

In [ ]:
# Select a numeric field for analysis
record_set_id = "summary_counts"
numeric_field = "count"
threshold = 5

filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold].copy()
print(f"Filtered rows with {numeric_field} > {threshold}:")
display(filtered_df)

if not filtered_df.empty and filtered_df[numeric_field].std() != 0:
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
else:
    filtered_df[f"{numeric_field}_normalized"] = 0.0

print(f"Normalized {numeric_field} for filtered rows:")
display(filtered_df[["section", numeric_field, f"{numeric_field}_normalized"]])

In [ ]:
# Derive a grouping variable for sections
def categorize_section(section_name: str) -> str:
    if "data_" in section_name or "preprocessing" in section_name or "maintenance" in section_name:
        return "curation_metadata"
    return "reference_metadata"

summary_counts_grouped = summary_counts.copy()
group_field = "section_group"
summary_counts_grouped[group_field] = summary_counts_grouped["section"].apply(categorize_section)

grouped_df = summary_counts_grouped.groupby(group_field, as_index=False)[numeric_field].mean()
print(f"Grouped data by {group_field}:")
display(grouped_df)

In [ ]:
# Additional metadata-focused processing: keyword lengths and topic tagging
keywords_df = dataframes["keywords"].copy()
keywords_df["keyword_length"] = keywords_df["keyword"].astype(str).str.len()

def keyword_topic(keyword: str) -> str:
    text = str(keyword).lower()
    if any(term in text for term in ["smiles", "fingerprints", "descriptor", "umap", "tanimoto"]):
        return "cheminformatics"
    if any(term in text for term in ["target", "bioactivity", "pchembl", "epigenetics", "inflammatory"]):
        return "bioactivity"
    if any(term in text for term in ["cosmetic", "skin", "toxicity", "sensitization"]):
        return "safety_application"
    return "other"

keywords_df["topic_group"] = keywords_df["keyword"].apply(keyword_topic)
dataframes["keywords_enriched"] = keywords_df

display(keywords_df.head(10))
display(keywords_df.groupby("topic_group", as_index=False)["keyword"].count().rename(columns={"keyword": "n_keywords"}))

## 5. Visualization
Visualize the extracted metadata to better understand the structure and emphasis of the dataset documentation.

In [ ]:
# Visualization 1: counts across metadata sections
plt.figure(figsize=(10, 5))
plot_df = summary_counts.sort_values("count", ascending=False)
sns.barplot(data=plot_df, x="section", y="count", palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Counts across Croissant metadata sections")
plt.xlabel("Metadata section")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: keyword topic distribution
keyword_topic_counts = (
    dataframes["keywords_enriched"]
    .groupby("topic_group", as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

plt.figure(figsize=(8, 4))
sns.barplot(data=keyword_topic_counts, x="topic_group", y="count", palette="magma")
plt.title("Keyword distribution by derived topic group")
plt.xlabel("Topic group")
plt.ylabel("Number of keywords")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: keyword length distribution
plt.figure(figsize=(8, 4))
sns.histplot(data=dataframes["keywords_enriched"], x="keyword_length", bins=10, kde=True, color="steelblue")
plt.title("Distribution of keyword lengths")
plt.xlabel("Keyword length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 6. Conclusion
This notebook loaded the INCITES Croissant schema with `mlcroissant` and explored the dataset metadata in a reproducible way.

### Key observations
- The dataset metadata loads successfully from the provided Croissant URL.
- The top-level dataset provides rich descriptive metadata, including authors, citations, distributions, keywords, use cases, biases, limitations, and preprocessing steps.
- In the current schema, no `recordSet` entities are exposed, so row-level loading with `dataset.records(record_set=<@id>)` is not available for this specific resource.
- Even without record sets, the Croissant document is useful for metadata-driven analysis and validation of dataset documentation.

### Recommended next steps
- Inspect the declared `distribution` entities by their `@id` values to locate downloadable files or companion resources.
- If an updated Croissant schema later exposes `recordSet` entities, reuse the included `@id`-based loading pattern to pull structured records directly into pandas.
- Extend this notebook by resolving distribution URLs into downloadable tables for deeper chemical and bioactivity analysis.

This workflow demonstrates a robust pattern for exploring Croissant datasets while strictly using `@id` identifiers whenever entities are available.